# Agents and Vector Databases - Exercise

In this notebook you will:
- load a small business knowledge base,
- use a vector database to retrieve relevant documents,
- build a simple ReAct-style agent that can call the retrieval tool.

Instructor note: the vector index is meant to be built once and then reused by students.

In [ ]:
# Install dependencies (run once)
!pip -q install sentence-transformers faiss-cpu openai

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(".")
VECTOR_DIR = BASE_DIR / "vector_store"
INDEX_PATH = VECTOR_DIR / "company_kb.index"
DOCS_PATH = VECTOR_DIR / "company_kb.json"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Instructor: set to True to build the index once
BUILD_INDEX = False

In [ ]:
documents = [
    {
        "doc_id": "DOC-001",
        "title": "Travel and Expense Policy",
        "category": "policy",
        "text": "Employees must submit travel expenses within 15 days of the trip. Receipts are required for purchases above 25 EUR. Hotel rates are capped at 180 EUR per night and the daily meal allowance is 45 EUR.",
    },
    {
        "doc_id": "DOC-002",
        "title": "Remote Work Guidelines",
        "category": "policy",
        "text": "Team members may work remotely up to three days per week. Core collaboration hours are 10:00 to 15:00 local time. A 300 EUR annual stipend is available for home office equipment.",
    },
    {
        "doc_id": "DOC-003",
        "title": "Return and Refund Policy",
        "category": "support",
        "text": "Customers can return hardware within 30 days of delivery. Opened items incur a 10 percent restocking fee. Refunds are issued within five business days after inspection.",
    },
    {
        "doc_id": "DOC-004",
        "title": "Support Tiers and SLA",
        "category": "support",
        "text": "Basic support responds within two business days. Growth support responds within 24 hours. Enterprise support responds within four hours and includes a 99.9 percent uptime target.",
    },
    {
        "doc_id": "DOC-005",
        "title": "Pricing Plans",
        "category": "product",
        "text": "The Starter plan is 99 EUR per month. The Growth plan is 399 EUR per month and includes priority support and monthly analytics. The Enterprise plan is custom priced and adds a BI dashboard and dedicated success manager.",
    },
    {
        "doc_id": "DOC-006",
        "title": "Data Retention and Backups",
        "category": "data",
        "text": "Daily backups are kept for 35 days and monthly snapshots are kept for 12 months. The recovery point objective is four hours and the recovery time objective is eight hours.",
    },
    {
        "doc_id": "DOC-007",
        "title": "Security Incident Response",
        "category": "security",
        "text": "Incidents are triaged within 60 minutes. The data protection officer must be notified within 24 hours for any confirmed personal data exposure. A postmortem is required within 10 days.",
    },
    {
        "doc_id": "DOC-008",
        "title": "API Usage Policy",
        "category": "product",
        "text": "Default rate limits are 120 requests per minute per API key. Batch endpoints allow up to 1,000 records per request. API keys must be rotated every 90 days.",
    },
    {
        "doc_id": "DOC-009",
        "title": "Product Roadmap Q3",
        "category": "product",
        "text": "Planned features include demand forecasting, supplier scorecards, and a configurable reorder point engine. The beta program starts in August.",
    },
    {
        "doc_id": "DOC-010",
        "title": "Marketing Campaign Q2",
        "category": "marketing",
        "text": "The Q2 campaign targets mid market retailers in the DACH region. The goal is 500 qualified leads with a 6 percent conversion rate. Primary channels are LinkedIn and industry newsletters.",
    },
    {
        "doc_id": "DOC-011",
        "title": "Cloud Vendor Contract Summary",
        "category": "operations",
        "text": "The cloud provider guarantees 99.9 percent availability and stores all customer data in the EU West region. Encryption at rest is mandatory and audit reports are delivered quarterly.",
    },
    {
        "doc_id": "DOC-012",
        "title": "Supply Chain Risk Memo",
        "category": "operations",
        "text": "Lithium price volatility is the main risk for Q3. Mitigation includes qualifying a second supplier and increasing safety stock to six weeks for critical components.",
    },
    {
        "doc_id": "DOC-013",
        "title": "Sustainability Report Highlights",
        "category": "sustainability",
        "text": "Operations are powered by 40 percent renewable energy. The 2027 goal is 70 percent renewable usage. The average CO2 emission per shipment is 1.2 kg.",
    },
    {
        "doc_id": "DOC-014",
        "title": "HR Onboarding Checklist",
        "category": "hr",
        "text": "New hires complete security training and product training within the first two weeks. Each employee is assigned a mentor for the first 90 days.",
    },
    {
        "doc_id": "DOC-015",
        "title": "Sales Playbook",
        "category": "sales",
        "text": "The ideal customer profile is a retailer with 50 to 500 employees and at least five locations. Key objections include integration effort and change management. Competitive differentiation is faster inventory turns.",
    },
    {
        "doc_id": "DOC-016",
        "title": "Customer Persona: SMB Retailer",
        "category": "marketing",
        "text": "Primary pain points are stockouts, limited analytics, and manual reordering. Buying triggers include a new ERP rollout and rapid store expansion.",
    },
    {
        "doc_id": "DOC-017",
        "title": "Finance Metrics Guide",
        "category": "finance",
        "text": "ARR is annual recurring revenue from subscriptions. Gross margin is revenue minus cost of goods sold, divided by revenue. Net revenue retention includes expansion and churn.",
    },
    {
        "doc_id": "DOC-018",
        "title": "Data Quality Guidelines",
        "category": "data",
        "text": "Automated checks must keep error rates below 3 percent and completeness above 97 percent. Failing pipelines must alert within 15 minutes.",
    },
    {
        "doc_id": "DOC-019",
        "title": "Churn Analysis Notes",
        "category": "finance",
        "text": "Top churn drivers are slow onboarding, low feature adoption, and price sensitivity. Accounts with weekly usage have half the churn rate of monthly users.",
    },
    {
        "doc_id": "DOC-020",
        "title": "AI Assistant Usage Policy",
        "category": "security",
        "text": "Human review is required for any external customer response generated by an AI assistant. Do not include confidential data in prompts. All prompts and outputs must be logged for audit.",
    },
]

# Preview the data
df_docs = pd.DataFrame(documents)
df_docs.head()

## Exercise 1: Explore the knowledge base

Tasks:
1. How many documents are in the dataset?
2. How many categories are there?
3. What is the average text length per category?
4. Print the three shortest documents by text length (doc_id and title).

In [ ]:
# TODO: number of documents
# TODO: number of categories
# TODO: average text length per category
# TODO: print three shortest documents (doc_id, title)

# Hint: you can create a length column with df_docs["text"].str.len()

## Exercise 2: Build or load the vector database

The vector index should be created once by the instructor and then reused by students.

Tasks:
1. If BUILD_INDEX is True, embed the documents and write the FAISS index and JSON metadata to disk.
2. If BUILD_INDEX is False, load the index and metadata from disk.

## Exercise 3: Vector search

Tasks:
1. Implement a function `vector_search(query, top_k=3)`.
2. Return a list of dictionaries that includes: rank, score, doc_id, title, category, and text.

In [ ]:
# Instructor-only: build the index once
if BUILD_INDEX:
    # TODO: create directory
    # TODO: build embeddings with SentenceTransformer
    # TODO: create FAISS index and add embeddings
    # TODO: save index to INDEX_PATH and docs to DOCS_PATH
    pass

# Student path: load the index and documents
# TODO: load index from INDEX_PATH
# TODO: load documents from DOCS_PATH
# TODO: initialize the embedding model for queries


def vector_search(query, top_k=3):
    # TODO: embed the query
    # TODO: search the index
    # TODO: format results with rank, score, and metadata
    return []

## Exercise 4: Try retrieval

Tasks:
1. Run three example queries and inspect the top results.
2. For each query, check whether the top document actually contains the answer.

## Exercise 5: Build a ReAct-style agent

Tasks:
1. Use an LLM with tool calling to access `vector_search`.
2. The agent should answer questions and cite the doc_id and title it used.
3. Test the agent with two business questions.

In [ ]:
example_queries = [
    "What is the reimbursement deadline for travel expenses?",
    "Which plan includes priority support and analytics?",
    "How long are backups kept and what is the RPO?",
]

# TODO: run vector_search for each query and print top titles

# --- ReAct-style agent ---
import os
from openai import OpenAI

# TODO: set your key, or load from environment
# os.environ["OPENAI_API_KEY"] = "..."

client = None
if os.getenv("OPENAI_API_KEY"):
    client = OpenAI()

# TODO: implement a tool-calling agent that uses vector_search
# Hint: use tools=[{"type": "function", "function": {...}}]
# and a short loop that calls the tool when requested.

def run_agent(question, max_steps=3):
    # TODO: handle the case where client is None
    pass

# Try:
# run_agent("Summarize the return policy and support SLA.")
# run_agent("What are the API limits and key rotation rules?")